In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torchmetrics import Accuracy
from tqdm.notebook import tqdm

In [ ]:
finetune_train=pd.read_csv(r'Finetune_Dataset/finetune_train_set.csv')
finetune_test=pd.read_csv(r'Finetune_Dataset/finetune_test_set.csv')

In [ ]:
ft_features_train=finetune_train.drop('label',axis=1)
ft_features_test=finetune_test.drop('label',axis=1)


ft_labels_train=finetune_train['label'].values
ft_labels_test=finetune_test['label'].values

In [ ]:
def img_reshape_std(train,test):
    img_train=[]
    img_test=[]

    transposed_train=train.transpose()
    transposed_test=test.transpose()


    for idx in range(len(train)):
        val=transposed_train[idx].values
        val=val.reshape(28,28)
        img_train.append(val)


    for idx in range(len(test)):
        val=transposed_test[idx].values
        val=val.reshape(28,28)
        img_test.append(val)


    return img_train,img_test

In [ ]:
ft_train_set_processed,ft_test_set_processed=img_reshape_std(ft_features_train,ft_features_test)

In [ ]:
ft_features_train_tensor=torch.tensor(ft_train_set_processed,dtype=torch.float32)
ft_features_test_tensor=torch.tensor(ft_test_set_processed,dtype=torch.float32)

ft_labels_train_tensor=torch.tensor(ft_labels_train,dtype=torch.long)
ft_labels_test_tensor=torch.tensor(ft_labels_test,dtype=torch.long)

In [ ]:
ft_tensorTrainDataset=TensorDataset(ft_features_train_tensor,ft_labels_train_tensor)
ft_tensorTestDataset=TensorDataset(ft_features_test_tensor,ft_labels_test_tensor)

In [ ]:
ft_trainloader=DataLoader(ft_tensorTrainDataset,batch_size=128,shuffle=True)
ft_testloader=DataLoader(ft_tensorTestDataset,batch_size=128,shuffle=False)

In [ ]:
def layer(in_channels,out_channels,kernel_size,padding,batchnorm=True,stride=(1,1)):
    if batchnorm == True:
        sub_model=nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    stride=stride),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU()
        )
        return sub_model
    else:
        sub_model=nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    stride=stride),
                    nn.ReLU()
        )
        return sub_model


In [ ]:
ft_model=nn.Sequential(
    layer(in_channels=1,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    layer(in_channels=16,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),

    nn.Dropout(0.2),

    layer(in_channels=16,out_channels=32,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


      layer(in_channels=32,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),   


    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


    layer(in_channels=64,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.Dropout(0.2),

    nn.AdaptiveAvgPool2d(1),

    nn.Flatten(),

    nn.Linear(64,10)
)

In [ ]:
ft_model

In [ ]:
ft_model.load_state_dict(torch.load('Weights/Number_Predict_Finall.pth'))

In [ ]:
# for ii in ft_model.parameters():
#    ii.requires_grad=False

In [ ]:
ft_model

In [ ]:
ft_model[12]=nn.Linear(64,7)

In [ ]:
ft_model

In [ ]:
if torch.cuda.is_available():
    device='cuda'
else:
    device='cpu'
device

In [ ]:
ft_model=ft_model.to(device)

In [ ]:
label_to_adx={
        'division':0,
        'addition':1,
        'equality':2,
        'subtraction':3,
        'X':4,
        'null':5,
        'number':6}

In [ ]:
ft_loss=nn.CrossEntropyLoss()

In [ ]:
ft_optimizer=optim.Adam(ft_model.parameters(),lr=0.02)

In [ ]:
c=0
for i in ft_model.parameters():
    if i.requires_grad==True:
        c+=i.numel()
print(c)

In [ ]:
class Average:
    def __init__(self):
        self.val=0
        self.sum=0
        self.avg=0
        self.count=0
        
    def update(self,value):
        self.val=value
        self.sum +=value
        self.count+=1
        self.avg=self.sum/self.count

In [ ]:
hist_loss_train_ft=[]
hist_acc_train_ft=[]

hist_loss_test_ft=[]
hist_acc_test_ft=[]

In [ ]:
def train_one_epoch(model, train_data_loader, loss, optimizer, epoch):
    model.train()
    total_loss_train = Average()
    total_acc_train = Accuracy(task='multiclass',num_classes=7).to(device)
    
    with tqdm(train_data_loader) as t_data_loader:
        for x_batch , y_batch in t_data_loader:
            x_batch=x_batch.unsqueeze(1)
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_predict = model(x_batch)
            loss_train = loss(y_predict.squeeze(), y_batch)
            total_acc_train(y_predict.squeeze(), y_batch.int())

            loss_train.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss_train.update(loss_train.item())

            t_data_loader.set_postfix(loss=total_loss_train.avg, acc=total_acc_train.compute().item())
            t_data_loader.set_description(f"Train epoch = {epoch}")

        hist_loss_train_ft.append(total_loss_train.avg)
        hist_acc_train_ft.append(total_acc_train.compute().item())

In [ ]:
def evaluate(model,test_data_loader,loss,epoch):
    model.eval()
    total_loss_test = Average()
    total_acc_test = Accuracy(task='multiclass',num_classes=7).to(device)
    
    with torch.no_grad():
        with tqdm(test_data_loader, colour="red") as t_data_loader:
            for x_batch, y_batch in t_data_loader:
                x_batch=x_batch.unsqueeze(1)
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                y_predict = model(x_batch)
                loss_test = loss(y_predict.squeeze(), y_batch)
                total_acc_test(y_predict.squeeze(), y_batch.int())

                total_loss_test.update(loss_test.item())

                t_data_loader.set_postfix(loss=total_loss_test.avg, acc=total_acc_test.compute().item())
                t_data_loader.set_description(f"Test epoch = {epoch}")
                
            hist_loss_test_ft.append(total_loss_test.avg)
            hist_acc_test_ft.append(total_acc_test.compute().item())

In [ ]:
c=0
for i in ft_model.parameters():
    if i.requires_grad==True:
        c+=i.numel()
print(c)

In [ ]:
epochs=5
for epoch in range(epochs):
    train_one_epoch(ft_model,ft_trainloader,ft_loss,ft_optimizer,epoch+1)
    evaluate(ft_model,ft_testloader,ft_loss,epoch+1)

In [ ]:
# Finetune: 
# Final Acc Train= 0.887, Final loss Train= 0.366        
# Final Acc Test= 0.857 , Final loss Test= 0.422
# Epochs= 105

In [ ]:
plt.plot(hist_loss_train_ft,label='Train Loss')
plt.plot(hist_loss_test_ft,label='Test Loss')
plt.grid(True)
plt.legend()

In [ ]:
plt.plot(hist_acc_train_ft,label='Train Acc')
plt.plot(hist_acc_test_ft,label='Test Acc')
plt.grid(True)
plt.legend()

In [ ]:
#torch.save(ft_model.state_dict(),'Finetune_Finall.pth')